# Accessing Data for ML

The purpose of this notebook is to provide minimal examples for loading protein gym data to your machine learning models. 

Since this tutorial goes over pytorch and sci-kit learn examples, but the packages are not necessary for ProteinGym you will have to add these packages to your environment first:

`uv pip install scikit-learn torch`

or `pip install` with your favorite environment manager.

In [ ]:
# Lets grab a dataset for showcasing

from proteingym.base import Dataset, Manifest

manifest = Manifest.from_path("../example_data/neime_2019.toml")
dataset = Dataset.from_manifest(manifest=manifest)

### 1: Extract All Sequences and Targets to a PyTorch Dataset

In [ ]:
# Example PyTorch dataset (requires torch)
class PG2TorchDataset:
    """PyTorch-compatible dataset wrapper for PG2 data."""

    def __init__(self, dataset: Dataset):
        self.sequences = []
        self.targets = []

        for assay in dataset.assays:
            for record in assay.records:
                self.sequences.append(record[0])
                for _, target_value in record[1].items():
                    self.targets.append(target_value)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]


pytorch_dataset = PG2TorchDataset(dataset)
print(f"Created PyTorch dataset with {len(pytorch_dataset)} samples")

if len(pytorch_dataset) > 0:
    sample_seq, sample_target = pytorch_dataset[0]
    print(f"Sample: {sample_seq}... -> {sample_target}")

if len(pytorch_dataset.sequences) > 0:
    print(
        f"Sequence lengths: min={min(len(s.value) for s in pytorch_dataset.sequences)}, max={max(len(s.value) for s in pytorch_dataset.sequences)}"
    )

### 2: Access Structural Information


In [ ]:
# Lets extend our PG2TorchDataset:


class StructurePG2TorchDataset(PG2TorchDataset):
    def __init__(self, dataset):
        self.x, self.y, self.z = [], [], []
        self.ref_structure = dataset.structures[0].value

        for model in self.ref_structure:
            for chain in model:
                for residue in chain:
                    for atom in residue:
                        x, y, z = atom.get_coord()
                        self.x.append(x)
                        self.y.append(y)
                        self.z.append(z)


pytorch_dataset = StructurePG2TorchDataset(dataset)
assert len(pytorch_dataset.x) == len(pytorch_dataset.y) == len(pytorch_dataset.z)
print(f"Found {len(pytorch_dataset.x)} atoms")

### Scikit-learn Compatible Format

In [ ]:
def to_sklearn_format(dataset, feature_extractor=None):
    """Convert PG2 dataset to scikit-learn format."""
    sequences = []
    targets = []

    for assay in dataset.assays:
        for record in assay.records:
            sequences.append(record[0])
            targets.append(record[1])

    if feature_extractor is None:
        # Simple feature extraction: sequence length and amino acid counts
        def simple_features(seq):
            seq = seq.value
            return [
                len(seq),
                seq.count("A"),
                seq.count("C"),
                seq.count("D"),
                seq.count("E"),
                seq.count("F"),
                seq.count("G"),
                seq.count("H"),
                seq.count("I"),
                seq.count("K"),
                seq.count("L"),
                seq.count("M"),
                seq.count("N"),
                seq.count("P"),
                seq.count("Q"),
                seq.count("R"),
                seq.count("S"),
                seq.count("T"),
                seq.count("V"),
                seq.count("W"),
                seq.count("Y"),
            ]

        feature_extractor = simple_features

    X = [feature_extractor(seq) for seq in sequences]
    y = targets

    return X, y


X, y = to_sklearn_format(dataset)
print(
    f"Created sklearn format: X shape = ({len(X)}, {len(X[0]) if X else 0}), y length = {len(y)}"
)

if X:
    print(f"Sample features: {X[0][:5]}... (first 5 features)")
    print(f"Sample targets: {y}")

## Summary

In this notebook, we've learned how to:

1. **Extract data for ML**: sequences, targets, and features
2. **Integrate with ML libraries** like PyTorch and scikit-learn

## Next Steps

Now you're ready to:
- Use PG2 Dataset in your own ML projects
- Create custom feature extractors for your specific needs
- Build reproducible protein engineering pipelines

Happy protein engineering! 🧬